In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score

In [ ]:
# Load the dataset using the pandas library function read_csv
data = pd.read_csv('e1_nutrients.csv')

In [ ]:
def plot_data(data):
    nutrient_cols = [c for c in data.columns if c != "Depth"]
    plt.figure(figsize=(10, 6))
    for col in nutrient_cols:
        plt.scatter(data["Depth"], data[col], s=35, alpha=0.75, label=col)

    plt.xlabel("Depth")
    plt.ylabel("Concentration")
    plt.title("Scatter Plot of All Nutrient Data vs Depth")
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_data(data)


In [ ]:
def plot_filter_comparison(unfiltered_data, filtered_data, variables_to_plot):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Impact of IQR Outlier Filtering by Depth', fontsize=18, fontweight='bold', y=0.98)

    for i, var in enumerate(variables_to_plot):
        row = i // 2
        col = i % 2
        ax = axes[row, col]
        
        ax.scatter(unfiltered_data['Depth'], unfiltered_data[var], 
                   c='tab:blue', alpha=0.6, s=30, label='Unfiltered Data')
        
        ax.scatter(filtered_data['Depth'], filtered_data[var], 
                   c='tab:orange', alpha=0.9, s=30, label='Filtered Data')
        
        ax.set_title(f'{var} vs Depth', fontsize=14, fontweight='bold')
        ax.set_xlabel('Depth (m)', fontsize=12)
        ax.set_ylabel('Concentration', fontsize=12)
        ax.grid(True, linestyle='--', alpha=0.7)
        ax.legend(loc='upper right')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

In [ ]:
# Function to filter out outliers based on the IQR method for a specific column
def filter_by_column(data, column_name):
    # Check if the inputted column exists in the provided data
    if column_name in data.columns:
        # Check if the required depth column exists in the provided data
        if "Depth" not in data.columns:
            # If the depth column does not exist, raise a ValueError with an appropriate message
            raise ValueError("Column 'Depth' does not exist in the Data.")
        
        # Directly copy the data to a new set without modifying the original data
        data_new = data.copy()

        # Sort the data, using the depth column to ensure that the quantile calculations are performed correctly for each depth level
        data_new = data_new.sort_values("Depth", kind="mergesort").reset_index(drop=True)

        # Calculate the first and third quartiles (Q1 and Q3) for the specified column, grouped by depth
        q1 = data_new.groupby("Depth")[column_name].quantile(0.25)
        q3 = data_new.groupby("Depth")[column_name].quantile(0.75)

        # Calculate the interquartile range (IQR) for the specified column, grouped by depth
        iqr = q3 - q1

        # Using the IQR, calculating the lower and upper bounds to detect outliers for the specified column, grouped by depth
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        # Using the pandas map function, apply the calculated lower and upper bounds to the original data based on the depth values, creating two new series that represent the lower and upper bounds for each row in the original data
        lower_mapped = data_new["Depth"].map(lower_bound)
        upper_mapped = data_new["Depth"].map(upper_bound)

        # Filter the original data to include only rows where the values in the specified column are within the calculated lower and upper bounds, grouped by depth
        filtered_data = data_new[
            (data_new[column_name] >= lower_mapped) & 
            (data_new[column_name] <= upper_mapped)
        ]
        
        # Reset the indexes of the filtered data to keep the sequential nature of the data intact after the outliers are removed
        filtered_data = filtered_data.reset_index(drop=True)
        
        # Return the filtered data, which contains only the rows where the values in the specified column are within the calculated bounds for each depth level
        return filtered_data
    
    # If the specified column does not exist in the provided data, raise a ValueError
    else:
        # If the column does not exist, raise a ValueError
        raise ValueError(f"Column '{column_name}' does not exist in the Data.")



In [ ]:
# Prepare the data for machine learning by separating the features (x) and the target variable (y), where the target variable is NITRITE


# Split the data into training and testing sets, using 80% of the data for training and 20% for testing
train_data, test_data = train_test_split(data, test_size=0.2, random_state=44)

columns_to_clean = ['NITRATE+NITRITE', 'AMMONIA', 'SILICATE', 'PHOSPHATE']
clean_train_data = train_data.copy()

for col in columns_to_clean:
    clean_train_data = filter_by_column(clean_train_data, col)

plot_data(clean_train_data)
plot_filter_comparison(train_data, clean_train_data, columns_to_clean)

x_train = clean_train_data.drop('NITRITE', axis=1)
y_train = clean_train_data['NITRITE']

x_test = test_data.drop('NITRITE', axis=1)
y_test = test_data['NITRITE']

scalar = RobustScaler()

x_train_scaled = scalar.fit_transform(x_train)
x_test_scaled = scalar.transform(x_test)


The dataset has been loaded into a pandas dataframe. Considering that the dataset containes no missing values, no data dropping steps were required for ones with missing data. Before formatting and outlier handling was carried out, the dataset was split into 2 sets, a training set (80%) and a testing set (20%). If the entire dataset had been filtered or scaled together, the test data could have been inadvertently leaked into the model's training phase. By isolating this part of the dataset, it can remain a completely unseen set, for an accurate real world simulation. 

Extreme values were handles using the data's Interquartile Range (IQR). Because natural nutrient concentrations vary drastically depending on how deep the water is, a global outlier filter would have incorrectly deleted natural deep-water readings. To solve this issue, the Pandas groupby function was used to calculate Q1, Q3 and the 1.5x IQR boundaries for each depth measurement. This filter was applied exclusively to the predictor variables to prevent skewing the models. 

To ensure the data was in a suitable format for the Neural Network and Linear Regression models, the predictor variables needed to be scaled. RobustScaler was selected for this task over standard methods like MinMaxScaler or StandardScaler. Robust Scalar centres the data using the median and scales the data using the interquartile range. Furthermore, due to the decision not to filter the target data, the natural extremes in the target data have been retained, RobustScaler ensures that any remaining extreme numbers do not disproportionately skew the variance of the normal, everyday chemical readings.

In [ ]:
lr_model = LinearRegression()
lr_model.fit(x_train_scaled, y_train)
lr_score_train = lr_model.score(x_train_scaled, y_train)
lr_score_test = lr_model.score(x_test_scaled, y_test)
print(f"Linear Regression R^2 Score (train): {lr_score_train}, R^2 Score (test): {lr_score_test}")

rf_model = RandomForestRegressor(random_state=44)
rf_model.fit(x_train_scaled, y_train)
rf_score_train = rf_model.score(x_train_scaled, y_train)
rf_score_test = rf_model.score(x_test_scaled, y_test)
print(f"Random Forest R^2 Score (train): {rf_score_train}, R^2 Score (test): {rf_score_test}")

nn_model = MLPRegressor(random_state=44, max_iter=1000)
nn_model.fit(x_train_scaled, y_train)
nn_score_train = nn_model.score(x_train_scaled, y_train)
nn_score_test = nn_model.score(x_test_scaled, y_test)
print(f"Neural Network R^2 Score (train): {nn_score_train}, R^2 Score (test): {nn_score_test}")

sample_input = x_test_scaled[[0]]
actual_value = y_test.iloc[0]

print(f"Actual NITRITE value: {actual_value}")
print(f"Linear Regression predicted: {lr_model.predict(sample_input)[0]}")
print(f"Random Forest predicted:     {rf_model.predict(sample_input)[0]}")
print(f"Neural Network predicted:    {nn_model.predict(sample_input)[0]}")

In [ ]:
lr_mean_squared_error = -cross_val_score(lr_model, x_train_scaled, y_train, cv=5, scoring='neg_mean_squared_error')
rf_mean_squared_error = -cross_val_score(rf_model, x_train_scaled, y_train, cv=5, scoring='neg_mean_squared_error')
nn_mean_squared_error = -cross_val_score(nn_model, x_train_scaled, y_train, cv=5, scoring='neg_mean_squared_error')
print(lr_mean_squared_error, rf_mean_squared_error, nn_mean_squared_error)

print("LR mean squared error: ", lr_mean_squared_error.mean())
print("RF mean squared error: ", rf_mean_squared_error.mean())
print("NN mean squared error: ", nn_mean_squared_error.mean())

In [ ]:
# Box Plot
plt.figure(figsize=(10, 6))
plt.boxplot([lr_mean_squared_error, rf_mean_squared_error, nn_mean_squared_error], labels=['Linear Regression', 'Random Forest', 'Neural Network'])
plt.ylabel('Mean Squared Error')
plt.title('Cross-Validated Mean Squared Error')
plt.show()